# Medical Symptom Chatbot — Full Pipeline
**Phase 1**: Generate dataset + Fine-tune RoBERTa  
**Phase 2**: Auto-label dialog dataset + Build retrieval index  
**Phase 3**: Launch Gradio demo

**Struktur folder yang dibutuhkan:**
```
project/
├── notebook.ipynb   ← file ini
├── data/
│   ├── iCliniq.json
│   ├── HealthCareMagic-100k.json
│   └── Healthcare_v2.csv  (dibuat otomatis)
└── outputs/         (dibuat otomatis)
```

## 0. Install dependencies

In [1]:
!pip install torch transformers gradio scikit-learn pandas numpy scipy matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports & Config

In [4]:
import os, json, pickle, re, random
import numpy as np
import pandas as pd
import torch
import scipy.sparse
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    RobertaTokenizer, RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ── paths ──────────────────────────────────────────────────────────────────
BASE_DIR  = os.getcwd()
DATA_DIR  = os.path.join(BASE_DIR, 'data')
SAVE_DIR  = os.path.join(BASE_DIR, 'outputs')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

CSV_PATH     = os.path.join(DATA_DIR, 'Healthcare_v2.csv')
ICLINIQ_PATH = os.path.join(DATA_DIR, 'iCliniq.json')
HCM_PATH     = os.path.join(DATA_DIR, 'HealthCareMagic-100k.json')

# ── hyperparams ────────────────────────────────────────────────────────────
MODEL_NAME  = 'roberta-base'
MAX_LEN     = 64
EPOCHS      = 5
BATCH_SIZE  = 16       # turunkan ke 8 jika RAM kurang
LR_BACKBONE = 2e-5
LR_HEAD     = 1e-4
MAX_HCM     = 30000    # jumlah dialog HCM yang dipakai

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device  : {device}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"Data    : {DATA_DIR}")
print(f"Outputs : {SAVE_DIR}")

Device  : cpu
Data    : d:\Tugas-tugas\Tugas kuliah Milan\Semester 6\Pemrosesan Bahasa Alami\UAS\Proyek Chatbot\data
Outputs : d:\Tugas-tugas\Tugas kuliah Milan\Semester 6\Pemrosesan Bahasa Alami\UAS\Proyek Chatbot\outputs


---
## PHASE 1 — Generate Dataset & Fine-tune RoBERTa

### 1.1 Generate Healthcare_v2.csv

In [3]:
SYMPTOMS_28 = [
    'chest pain','dizziness','sore throat','sneezing','weight loss','blurred vision',
    'nausea','vomiting','sweating','insomnia','diarrhea','depression','weight gain',
    'rash','swelling','headache','fever','fatigue','abdominal pain','muscle pain',
    'back pain','runny nose','tremors','anxiety','cough','joint pain',
    'shortness of breath','appetite loss'
]

DISEASE_PROFILES = {
    'Allergy':                {'p':['sneezing','runny nose'],              's':['rash','swelling','cough','headache','fatigue']},
    'Anemia':                 {'p':['fatigue','appetite loss'],            's':['dizziness','shortness of breath','weight loss','headache']},
    'Anxiety':                {'p':['anxiety','insomnia'],                 's':['sweating','tremors','headache','chest pain','fatigue']},
    'Arthritis':              {'p':['joint pain','swelling'],              's':['muscle pain','back pain','fatigue','fever']},
    'Asthma':                 {'p':['shortness of breath','cough'],        's':['chest pain','fatigue','sweating','anxiety']},
    'Bronchitis':             {'p':['cough','chest pain'],                 's':['shortness of breath','fatigue','fever','sore throat']},
    'COVID-19':               {'p':['fever','cough'],                      's':['shortness of breath','fatigue','muscle pain','sore throat']},
    'Chronic Kidney Disease': {'p':['swelling','back pain'],               's':['fatigue','shortness of breath','appetite loss','nausea']},
    'Common Cold':            {'p':['runny nose','sore throat'],           's':['sneezing','cough','fever','headache','fatigue']},
    'Dementia':               {'p':['depression','insomnia'],              's':['fatigue','appetite loss','headache','dizziness']},
    'Depression':             {'p':['depression','fatigue'],               's':['insomnia','appetite loss','weight loss','anxiety']},
    'Dermatitis':             {'p':['rash','swelling'],                    's':['fatigue','fever','appetite loss','headache']},
    'Diabetes':               {'p':['weight loss','blurred vision'],       's':['fatigue','sweating','weight gain','dizziness']},
    'Epilepsy':               {'p':['tremors','dizziness'],                's':['headache','fatigue','muscle pain','anxiety']},
    'Food Poisoning':         {'p':['vomiting','diarrhea'],                's':['nausea','abdominal pain','fever','fatigue']},
    'Gastritis':              {'p':['abdominal pain','nausea'],            's':['vomiting','appetite loss','fever','back pain']},
    'Heart Disease':          {'p':['chest pain','shortness of breath'],   's':['fatigue','swelling','back pain','dizziness']},
    'Hypertension':           {'p':['headache','dizziness'],               's':['chest pain','fatigue','shortness of breath','blurred vision']},
    'IBS':                    {'p':['abdominal pain','diarrhea'],          's':['nausea','fatigue','weight loss','appetite loss']},
    'Influenza':              {'p':['fever','muscle pain'],                's':['cough','sore throat','fatigue','headache']},
    'Liver Disease':          {'p':['abdominal pain','fatigue'],           's':['nausea','weight loss','appetite loss','vomiting']},
    'Migraine':               {'p':['headache','nausea'],                  's':['dizziness','blurred vision','fatigue','vomiting']},
    'Obesity':                {'p':['weight gain','fatigue'],              's':['joint pain','shortness of breath','back pain','sweating']},
    "Parkinson's":            {'p':['tremors','muscle pain'],              's':['depression','fatigue','dizziness','insomnia']},
    'Pneumonia':              {'p':['fever','shortness of breath'],        's':['cough','chest pain','fatigue','sweating']},
    'Sinusitis':              {'p':['headache','runny nose'],              's':['sore throat','fever','fatigue','cough']},
    'Stroke':                 {'p':['dizziness','blurred vision'],         's':['headache','muscle pain','fatigue','nausea']},
    'Thyroid Disorder':       {'p':['weight loss','insomnia'],             's':['weight gain','fatigue','sweating','tremors']},
    'Tuberculosis':           {'p':['cough','weight loss'],                's':['fever','fatigue','appetite loss','sweating']},
    'Ulcer':                  {'p':['abdominal pain','vomiting'],          's':['nausea','back pain','appetite loss','fatigue']},
}

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded existing dataset: {df.shape}")
else:
    random.seed(42); np.random.seed(42)
    rows, pid = [], 1
    all_diseases = list(DISEASE_PROFILES.keys())
    for disease, profile in DISEASE_PROFILES.items():
        primary, secondary = profile['p'], profile['s']
        other = [s for s in SYMPTOMS_28 if s not in primary and s not in secondary]
        for _ in range(833):
            syms = list(primary)
            syms += random.sample(secondary, random.randint(1, min(3, len(secondary))))
            if random.random() < 0.4:
                syms += random.sample(other, 1)
            random.shuffle(syms)
            label = disease if random.random() > 0.05 else random.choice(
                [d for d in all_diseases if d != disease])
            rows.append({'Patient_ID': pid,
                         'Age': random.randint(18, 85),
                         'Gender': random.choice(['Male','Female','Other']),
                         'Symptoms': ', '.join(syms),
                         'Symptom_Count': len(syms),
                         'Disease': label})
            pid += 1
    df = pd.DataFrame(rows)
    df.to_csv(CSV_PATH, index=False)
    print(f"Generated dataset: {df.shape}")

print(f"Diseases : {df['Disease'].nunique()}")
print(f"Balance  : min={df['Disease'].value_counts().min()}, max={df['Disease'].value_counts().max()}")
df.head(3)

Loaded existing dataset: (25000, 6)
Diseases : 30
Balance  : min=818, max=849


,Patient_ID,Age,Gender,Symptoms,Symptom_Count,Disease
0,1,45,Male,"fatigue, sweating, rash, cough, sneezing, runn...",6,Arthritis
1,2,18,Male,"fatigue, runny nose, sneezing, rash, cough, ap...",6,Allergy
2,3,51,Male,"headache, runny nose, swelling, cough, depress...",6,Allergy


### 1.2 Label mapping & train/val/test split (80/10/10)

In [4]:
labels_list = sorted(df['Disease'].unique())
label2id    = {l: i for i, l in enumerate(labels_list)}
id2label    = {i: l for l, i in label2id.items()}
NUM_LABELS  = len(labels_list)

df['label'] = df['Disease'].map(label2id)
df['text']  = df['Symptoms'].apply(
    lambda x: ', '.join(sorted([s.strip() for s in x.split(',')])))

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 20000 | Val: 2500 | Test: 2500


### 1.3 Tokenizer & Dataset

In [5]:
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

class SymptomDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = list(texts)
        self.labels = list(labels)
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], truncation=True, max_length=MAX_LEN,
                        padding='max_length', return_tensors='pt')
        return {'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(self.labels[idx], dtype=torch.long)}

train_ds = SymptomDataset(train_df['text'], train_df['label'])
val_ds   = SymptomDataset(val_df['text'],   val_df['label'])
test_ds  = SymptomDataset(test_df['text'],  test_df['label'])
print("Datasets ready.")

def evaluate(model, dataset, batch_size=64):
    loader = DataLoader(dataset, batch_size=batch_size)
    preds, labels = [], []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            out = model(input_ids=batch['input_ids'].to(device),
                        attention_mask=batch['attention_mask'].to(device))
            preds.extend(out.logits.argmax(dim=-1).cpu().numpy())
            labels.extend(batch['labels'].numpy())
    return np.array(preds), np.array(labels)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

C:\Users\Milan\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Milan\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

Datasets ready.


### 1.4 Baseline — Base RoBERTa (no fine-tuning)

In [6]:
print("Evaluating BASE RoBERTa...")
base_model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id).to(device)
base_preds, base_labels_arr = evaluate(base_model, test_ds)
base_acc = accuracy_score(base_labels_arr, base_preds)
base_f1  = f1_score(base_labels_arr, base_preds, average='macro')
print(f"Base Accuracy : {base_acc:.4f}")
print(f"Base Macro F1 : {base_f1:.4f}")
del base_model
if torch.cuda.is_available(): torch.cuda.empty_cache()

Evaluating BASE RoBERTa...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Base Accuracy : 0.0356
Base Macro F1 : 0.0035


### 1.5 Fine-tune RoBERTa

In [7]:
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64)

optimizer = AdamW([
    {'params': model.roberta.parameters(),    'lr': LR_BACKBONE},
    {'params': model.classifier.parameters(), 'lr': LR_HEAD},
], weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(optimizer, int(0.1*total_steps), total_steps)

model.to(device)
best_val_f1 = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        out  = model(input_ids=batch['input_ids'].to(device),
                     attention_mask=batch['attention_mask'].to(device),
                     labels=batch['labels'].to(device))
        loss = out.loss
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()
        if step % 100 == 0:
            print(f"  Epoch {epoch+1} | Step {step:>3}/{len(train_loader)} | Loss: {loss.item():.4f}")

    val_preds, val_lbls = [], []
    model.eval()
    with torch.no_grad():
        for batch in val_loader:
            out = model(input_ids=batch['input_ids'].to(device),
                        attention_mask=batch['attention_mask'].to(device))
            val_preds.extend(out.logits.argmax(dim=-1).cpu().numpy())
            val_lbls.extend(batch['labels'].numpy())
    val_f1  = f1_score(val_lbls, val_preds, average='macro')
    val_acc = accuracy_score(val_lbls, val_preds)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} "
          f"| Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        ckpt = os.path.join(SAVE_DIR, 'best_checkpoint')
        model.save_pretrained(ckpt); tokenizer.save_pretrained(ckpt)
        print(f"  → Best model saved (F1: {val_f1:.4f})")

print(f"\nTraining done. Loading best checkpoint (F1: {best_val_f1:.4f})...")
model = RobertaForSequenceClassification.from_pretrained(
    os.path.join(SAVE_DIR, 'best_checkpoint')).to(device)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Epoch 1 | Step   0/1250 | Loss: 3.3687
  Epoch 1 | Step 100/1250 | Loss: 3.3574
  Epoch 1 | Step 200/1250 | Loss: 2.4051
  Epoch 1 | Step 300/1250 | Loss: 1.4751
  Epoch 1 | Step 400/1250 | Loss: 0.7290
  Epoch 1 | Step 500/1250 | Loss: 0.6295
  Epoch 1 | Step 600/1250 | Loss: 0.6309
  Epoch 1 | Step 700/1250 | Loss: 0.8209
  Epoch 1 | Step 800/1250 | Loss: 0.2493
  Epoch 1 | Step 900/1250 | Loss: 0.2038
  Epoch 1 | Step 1000/1250 | Loss: 0.8711
  Epoch 1 | Step 1100/1250 | Loss: 0.5881
  Epoch 1 | Step 1200/1250 | Loss: 0.2758
Epoch 1/5 | Loss: 1.2596 | Val Acc: 0.8544 | Val F1: 0.8523
  → Best model saved (F1: 0.8523)
  Epoch 2 | Step   0/1250 | Loss: 1.1364
  Epoch 2 | Step 100/1250 | Loss: 0.5758
  Epoch 2 | Step 200/1250 | Loss: 0.2509
  Epoch 2 | Step 300/1250 | Loss: 0.8159
  Epoch 2 | Step 400/1250 | Loss: 0.6485
  Epoch 2 | Step 500/1250 | Loss: 0.3498
  Epoch 2 | Step 600/1250 | Loss: 1.0127
  Epoch 2 | Step 700/1250 | Loss: 0.9814
  Epoch 2 | Step 800/1250 | Loss: 0.6581
 

### 1.6 Evaluate on test set + save artifacts

In [8]:
ft_preds, ft_labels_arr = evaluate(model, test_ds)
ft_acc  = accuracy_score(ft_labels_arr, ft_preds)
ft_f1   = f1_score(ft_labels_arr, ft_preds, average='macro')
ft_prec = precision_score(ft_labels_arr, ft_preds, average='macro', zero_division=0)
ft_rec  = recall_score(ft_labels_arr, ft_preds, average='macro', zero_division=0)

print(f"\n{'Metric':<15} {'Base':>10} {'Fine-tuned':>12}  {'Delta':>8}")
print('-'*50)
print(f"{'Accuracy':<15} {base_acc:>10.4f} {ft_acc:>12.4f}  {ft_acc-base_acc:>+8.4f}")
print(f"{'Macro F1':<15} {base_f1:>10.4f} {ft_f1:>12.4f}  {ft_f1-base_f1:>+8.4f}")
print(f"{'Macro Prec':<15} {'-':>10} {ft_prec:>12.4f}")
print(f"{'Macro Recall':<15} {'-':>10} {ft_rec:>12.4f}")

# Confusion matrix
cm = confusion_matrix(ft_labels_arr, ft_preds)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=False, cmap='Blues',
            xticklabels=labels_list, yticklabels=labels_list)
plt.title('Confusion Matrix — Fine-tuned RoBERTa')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix saved.")

# Save model artifacts
model_save = os.path.join(SAVE_DIR, 'roberta_finetuned')
model.save_pretrained(model_save); tokenizer.save_pretrained(model_save)
with open(os.path.join(model_save, 'label_mappings.json'), 'w') as f:
    json.dump({'label2id': label2id,
               'id2label': {str(k): v for k, v in id2label.items()},
               'labels_list': labels_list}, f, indent=2)

metrics = {'base':     {'accuracy': float(base_acc), 'f1_macro': float(base_f1)},
           'finetuned':{'accuracy': float(ft_acc),   'f1_macro': float(ft_f1),
                        'precision_macro': float(ft_prec), 'recall_macro': float(ft_rec)}}
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

report = classification_report(ft_labels_arr, ft_preds, target_names=labels_list)
with open(os.path.join(SAVE_DIR, 'classification_report.txt'), 'w') as f:
    f.write(report)
print(report)
print("Phase 1 done.")


Metric                Base   Fine-tuned     Delta
--------------------------------------------------
Accuracy            0.0356       0.8668   +0.8312
Macro F1            0.0035       0.8662   +0.8627
Macro Prec               -       0.8753
Macro Recall             -       0.8664


C:\Users\Milan\AppData\Local\Temp\ipykernel_23412\2155636360.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Confusion matrix saved.
                        precision    recall  f1-score   support

               Allergy       0.93      0.95      0.94        85
                Anemia       0.88      0.90      0.89        82
               Anxiety       0.99      0.91      0.95        82
             Arthritis       0.90      0.96      0.93        84
                Asthma       0.79      0.89      0.84        84
            Bronchitis       0.81      0.82      0.82        84
              COVID-19       0.68      0.81      0.74        83
Chronic Kidney Disease       0.96      0.94      0.95        85
           Common Cold       0.88      0.84      0.86        82
              Dementia       0.91      0.83      0.87        82
            Depression       0.82      0.92      0.87        85
            Dermatitis       0.95      0.99      0.97        83
              Diabetes       0.93      0.92      0.92        85
              Epilepsy       0.83      0.93      0.88        83
        Food Po

---
## PHASE 2 — Auto-label Dialogs & Build Retrieval Index

### 2.1 Symptom extraction helpers

In [12]:
SYMPTOM_SYNONYMS = {
    'fever':               ['fever','feverish','high temperature'],
    'cough':               ['cough','coughing'],
    'headache':            ['headache','head pain'],
    'dizziness':           ['dizz','vertigo','lightheaded','spinning'],
    'nausea':              ['nausea','nauseous','nauseated','queasy'],
    'vomiting':            ['vomit','throwing up','threw up'],
    'fatigue':             ['fatigue','tired','exhausted','weakness','weak'],
    'shortness of breath': ['shortness of breath','difficulty breath','breathless'],
    'chest pain':          ['chest pain','chest ache','chest tightness'],
    'abdominal pain':      ['abdominal pain','stomach pain','belly pain','stomach ache'],
    'diarrhea':            ['diarrhea','loose stool'],
    'rash':                ['rash','skin rash','hives'],
    'swelling':            ['swelling','swollen','edema'],
    'depression':          ['depress','hopeless','low mood'],
    'anxiety':             ['anxiety','anxious','worried','panic'],
    'insomnia':            ['insomnia','trouble sleeping','sleepless','cannot sleep'],
    'joint pain':          ['joint pain','joint ache','stiff joint'],
    'muscle pain':         ['muscle pain','muscle ache','body ache','body pain'],
    'back pain':           ['back pain','backache','lower back'],
    'weight loss':         ['weight loss','losing weight','lost weight'],
    'weight gain':         ['weight gain','gaining weight'],
    'blurred vision':      ['blurred vision','blurry vision','double vision'],
    'tremors':             ['tremor','shaking','shiver','trembling'],
    'appetite loss':       ['appetite loss','loss of appetite','no appetite'],
    'sore throat':         ['sore throat','throat pain','throat ache'],
    'sneezing':            ['sneez'],
    'runny nose':          ['runny nose','nasal discharge','stuffy nose'],
    'sweating':            ['sweating','sweat','night sweat'],
}

def extract_symptoms(text):
    text_lower = text.lower()
    found = set()
    for sym, variants in SYMPTOM_SYNONYMS.items():
        for v in variants:
            if v in text_lower:
                found.add(sym); break
    return list(found)

# Test
print(extract_symptoms("I have fever, feel dizzy and nauseous"))

['nausea', 'dizziness', 'fever']


### 2.1b Tambah keyword Bahasa Indonesia (bilingual support)

In [42]:
# Keyword Bahasa Indonesia — gejala dasar
INDO_KEYWORDS = {
    'fever':               ['demam', 'panas tinggi', 'badan panas', 'meriang'],
    'cough':               ['batuk'],
    'headache':            ['sakit kepala', 'kepala sakit', 'nyeri kepala', 'pening'],
    'dizziness':           ['pusing', 'kepala berputar', 'kliyengan', 'vertigo'],
    'nausea':              ['mual', 'eneg', 'enek'],
    'vomiting':            ['muntah'],
    'fatigue':             ['lemas', 'lelah', 'capek', 'kelelahan', 'tidak bertenaga', 'letih'],
    'shortness of breath': ['sesak napas', 'susah bernapas', 'napas sesak', 'sesak nafas'],
    'chest pain':          ['sakit dada', 'nyeri dada', 'dada sakit', 'dada nyeri'],
    'abdominal pain':      ['sakit perut', 'nyeri perut', 'perut sakit', 'mulas', 'perih lambung'],
    'diarrhea':            ['diare', 'mencret', 'bab cair'],
    'rash':                ['ruam', 'gatal', 'bintik merah', 'biduran', 'bentol'],
    'swelling':            ['bengkak', 'pembengkakan', 'membengkak'],
    'depression':          ['depresi', 'sedih berkepanjangan', 'putus asa', 'murung'],
    'anxiety':             ['cemas', 'gelisah', 'khawatir', 'panik', 'was-was'],
    'insomnia':            ['susah tidur', 'tidak bisa tidur', 'sulit tidur', 'insomnia', 'sukar tidur'],
    'joint pain':          ['nyeri sendi', 'sakit sendi', 'sendi sakit', 'ngilu sendi'],
    'muscle pain':         ['nyeri otot', 'sakit otot', 'pegal', 'pegal-pegal', 'otot sakit'],
    'back pain':           ['sakit punggung', 'nyeri punggung', 'punggung sakit', 'sakit pinggang'],
    'weight loss':         ['berat badan turun', 'bb turun', 'kurus mendadak', 'berat turun'],
    'weight gain':         ['berat badan naik', 'bb naik', 'gemuk', 'berat naik'],
    'blurred vision':      ['penglihatan kabur', 'mata kabur', 'pandangan kabur', 'buram'],
    'tremors':             ['gemetar', 'tangan gemetar', 'badan gemetar', 'tremor'],
    'appetite loss':       ['tidak nafsu makan', 'nafsu makan hilang', 'tidak mau makan', 'kehilangan nafsu makan'],
    'sore throat':         ['sakit tenggorokan', 'tenggorokan sakit', 'radang tenggorokan', 'nyeri tenggorokan'],
    'sneezing':            ['bersin'],
    'runny nose':          ['pilek', 'hidung meler', 'ingus', 'hidung tersumbat'],
    'sweating':            ['berkeringat', 'keringat berlebih', 'keringat dingin', 'banyak keringat'],
}

# Tambahan: nama penyakit umum → gejala representatif (extend, bukan overwrite)
DISEASE_NAME_HINTS = {
    'fever':         ['flu', 'demam berdarah', 'dbd'],
    'cough':         ['flu', 'batuk pilek'],
    'runny nose':    ['flu', 'pilek'],
    'abdominal pain':['maag', 'asam lambung', 'gerd'],
    'headache':      ['migrain', 'sinusitis'],
    'dizziness':     ['vertigo'],
    'joint pain':    ['rematik', 'asam urat'],
    'anxiety':       ['panic attack'],
}
for sym, kws in DISEASE_NAME_HINTS.items():
    INDO_KEYWORDS.setdefault(sym, []).extend(kws)

for sym, kws in INDO_KEYWORDS.items():
    if sym in SYMPTOM_SYNONYMS:
        SYMPTOM_SYNONYMS[sym].extend(kws)
    else:
        SYMPTOM_SYNONYMS[sym] = kws

# Terjemahan nama gejala (untuk display & follow-up question)
SYMPTOM_ID = {
    'fever': 'demam', 'cough': 'batuk', 'headache': 'sakit kepala',
    'dizziness': 'pusing', 'nausea': 'mual', 'vomiting': 'muntah',
    'fatigue': 'lemas', 'shortness of breath': 'sesak napas',
    'chest pain': 'nyeri dada', 'abdominal pain': 'sakit perut',
    'diarrhea': 'diare', 'rash': 'ruam/gatal', 'swelling': 'bengkak',
    'depression': 'depresi', 'anxiety': 'cemas/gelisah',
    'insomnia': 'susah tidur', 'joint pain': 'nyeri sendi',
    'muscle pain': 'nyeri otot/pegal', 'back pain': 'sakit punggung',
    'weight loss': 'berat badan turun', 'weight gain': 'berat badan naik',
    'blurred vision': 'penglihatan kabur', 'tremors': 'gemetar',
    'appetite loss': 'tidak nafsu makan', 'sore throat': 'sakit tenggorokan',
    'sneezing': 'bersin', 'runny nose': 'pilek', 'sweating': 'berkeringat',
}

print("Bilingual keywords added.")
print("Test EN:", extract_symptoms("I have fever and feel dizzy"))
print("Test ID:", extract_symptoms("saya demam dan pusing"))
print("Test mix:", extract_symptoms("saya fever dan sakit perut"))

# ════════════════════════════════════════════════════════════════════
# Hybrid language detection: keyword domain + langdetect fallback
# ════════════════════════════════════════════════════════════════════
try:
    from langdetect import detect as _ld_detect, DetectorFactory
    DetectorFactory.seed = 42
    _HAS_LANGDETECT = True
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "langdetect", "-q"])
    try:
        from langdetect import detect as _ld_detect, DetectorFactory
        DetectorFactory.seed = 42
        _HAS_LANGDETECT = True
    except Exception:
        _HAS_LANGDETECT = False

_INDO_KW = {'saya','aku','gua','gue','dan','merasa','sakit','tidak','dok','kamu','iya',
            'engga','sejak','juga','nyeri','dada','perut','kepala','gejala','demam',
            'pusing','mual','batuk','sesak','lemas','dengan','yang','ini','itu','sudah',
            'apakah','kenapa','bagaimana','tolong','bisa','mau','ingin','nih','ya',
            'halo','hai','pagi','siang','sore','malam','selamat'}
_ENG_KW  = {'i','have','feel','my','and','the','with','since','also','pain','fever',
            'cough','dizzy','sick','doctor','chest','head','stomach','am','is','was',
            'do','you','can','what','why','how','please','help','having','been'}

def detect_lang_smart(text, prev='en'):
    """Return ('id'|'en', method). Keyword domain first, then langdetect, then prev."""
    t = text.lower(); words = t.split()
    id_hits = sum(1 for w in words if w in _INDO_KW)
    en_hits = sum(1 for w in words if w in _ENG_KW)
    if id_hits > en_hits and id_hits >= 1:
        return 'id', f'keyword(id={id_hits},en={en_hits})'
    if en_hits > id_hits and en_hits >= 1:
        return 'en', f'keyword(id={id_hits},en={en_hits})'
    if len(words) >= 2 and _HAS_LANGDETECT:
        try:
            return ('id' if _ld_detect(text)=='id' else 'en'), 'langdetect'
        except Exception:
            return prev, 'fallback'
    return prev, 'too_short'

print("Language detection ready. langdetect:", _HAS_LANGDETECT)
print("Test:", detect_lang_smart("saya merasa dizzy dan mual"))


Bilingual keywords added.
Test EN: ['dizziness', 'fever']
Test ID: ['dizziness', 'fever']
Test mix: ['abdominal pain', 'fever']
Language detection ready. langdetect: True
Test: ('id', 'keyword(id=4,en=1)')


### 2.2 Load dialog datasets

In [18]:
records = []

if os.path.exists(ICLINIQ_PATH):
    with open(ICLINIQ_PATH) as f: data = json.load(f)
    for item in data:
        p = item.get('input','').strip(); d = item.get('answer_icliniq','').strip()
        if p and d: records.append({'source':'icliniq','patient_text':p,'doctor_text':d})
    print(f"iCliniq: {len(data)} dialogs")
else:
    print(f"WARNING: {ICLINIQ_PATH} not found")

if os.path.exists(HCM_PATH):
    with open(HCM_PATH) as f: data = json.load(f)
    if MAX_HCM: data = data[:MAX_HCM]
    for item in data:
        p = item.get('input','').strip(); d = item.get('output','').strip()
        if p and d: records.append({'source':'hcm','patient_text':p,'doctor_text':d})
    print(f"HealthCareMagic: {len(data)} dialogs")
else:
    print(f"WARNING: {HCM_PATH} not found")

dialog_df = pd.DataFrame(records)
print(f"Total: {len(dialog_df)} dialogs")

NameError: name 'os' is not defined

### 2.3 Auto-label dialogs using trained classifier

In [9]:
class TextDS(Dataset):
    def __init__(self, texts):
        self.texts = list(texts)
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        t    = self.texts[idx]
        syms = extract_symptoms(t)
        inp  = ', '.join(sorted(syms)) if syms else t[:128]
        enc  = tokenizer(inp, truncation=True, max_length=MAX_LEN,
                         padding='max_length', return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0)}

def batch_classify(texts, batch_size=128):
    ds = TextDS(texts); loader = DataLoader(ds, batch_size=batch_size)
    diseases, confs = [], []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            out   = model(input_ids=batch['input_ids'].to(device),
                          attention_mask=batch['attention_mask'].to(device))
            probs = torch.softmax(out.logits, dim=-1)
            c, idx = probs.max(dim=-1)
            diseases.extend([id2label[int(i)] for i in idx])
            confs.extend(c.cpu().tolist())
    return diseases, confs

print(f"Auto-labeling {len(dialog_df)} dialogs...")
diseases, confs = batch_classify(dialog_df['patient_text'].tolist())
dialog_df['predicted_disease'] = diseases
dialog_df['confidence']        = confs
print("Done.")
print(dialog_df['predicted_disease'].value_counts().head(8).to_string())

NameError: name 'Dataset' is not defined

### 2.4 Build TF-IDF retrieval index + disease-symptom stats

In [8]:
# TF-IDF
tfidf        = TfidfVectorizer(max_features=10000, ngram_range=(1,2), stop_words='english')
tfidf_matrix = tfidf.fit_transform(dialog_df['patient_text'])
print(f"TF-IDF matrix: {tfidf_matrix.shape}")

def trim(text, n=3):
    return ' '.join(re.split(r'(?<=[.!?]) +', text.strip())[:n])
dialog_df['doctor_text_short'] = dialog_df['doctor_text'].apply(trim)

# Disease-symptom co-occurrence stats
disease_symptom_stats = {}
for disease in labels_list:
    rows = df[df['Disease'] == disease]
    stats = {}
    for sym in SYMPTOMS_28:
        count = rows['Symptoms'].str.contains(re.escape(sym), case=False).sum()
        stats[sym] = count / max(len(rows), 1)
    disease_symptom_stats[disease] = stats

# Save all Phase 2 artifacts
dialog_df.to_csv(os.path.join(SAVE_DIR, 'dialog_index.csv'), index=False)
with open(os.path.join(SAVE_DIR, 'tfidf_vectorizer.pkl'), 'wb') as f:
    pickle.dump(tfidf, f)
scipy.sparse.save_npz(os.path.join(SAVE_DIR, 'tfidf_matrix.npz'), tfidf_matrix)
with open(os.path.join(SAVE_DIR, 'disease_symptom_stats.json'), 'w') as f:
    json.dump(disease_symptom_stats, f)

print("Phase 2 artifacts saved.")
print("\nContents of outputs/:")
for fn in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, fn))
    print(f"  {fn:<40} {size/1024/1024:.1f} MB")

NameError: name 'TfidfVectorizer' is not defined

---
## PHASE 3 — MedicalChatbot + Gradio Demo

### 3.1 Load base model for comparison

In [43]:
from transformers import RobertaForSequenceClassification

base_model_demo = RobertaForSequenceClassification.from_pretrained(
    'roberta-base', num_labels=NUM_LABELS,
    id2label=id2label, label2id=label2id).to(device)
base_model_demo.eval()
print("Base model loaded for comparison.")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Base model loaded for comparison.


In [33]:
import re
from sklearn.metrics.pairwise import cosine_similarity

### 3.2 MedicalChatbot class

In [ ]:
# ════════════════════════════════════════════════════════════════════
# MedicalChatbot — Multi-layer conversational agent (bilingual)
# Layers: intent detection · off-topic guard · state manager ·
#         symptom gathering · health Q&A · red-flag detection
# ════════════════════════════════════════════════════════════════════
CONFIDENCE_THRESHOLD = 0.70
MAX_TURNS = 5

GREETING = {'halo','hai','hi','hello','hey','pagi','siang','sore','malam',
            'assalamualaikum','permisi','selamat','help','tolong'}
YES = {'iya','ya','ada','benar','betul','yoi','yup','yes','yep','y','iyaa',
       'betul dok','ada dok','iya dok','ho oh','hooh','yoa'}
NO  = {'tidak','ngga','nggak','enggak','gak','engga','no','nope','tdk','bukan',
       'ga','gada','tidak ada','ga ada','engga dok','enggak dok'}
THANKS = {'terima kasih','makasih','thanks','thank you','thx','tengkyu','makasi','trims'}
BYE = {'bye','dadah','sampai jumpa','selesai','udah cukup','keluar','exit','udahan'}
HEALTH_CTX = {'sakit','nyeri','gejala','penyakit','dokter','obat','demam','sehat',
              'kesehatan','badan','tubuh','sick','pain','disease','doctor','medicine',
              'health','hurt','feel','rasa','merasa','keluhan','mengalami','kondisi',
              'flu','pilek','batuk','meriang','masuk angin','virus','infeksi','radang',
              'alergi','mual','pusing','lemas','capek','demam','diare','sembuh'}
# Red-flag: gejala/kata yang butuh perhatian darurat
RED_FLAG_WORDS = {'pingsan','tidak sadar','kejang','pendarahan hebat','sesak berat',
                  'nyeri dada hebat','sulit bernapas','unconscious','seizure','severe bleeding'}


class MedicalChatbot:
    def __init__(self, lang_mode='auto'):
        self.lang_mode = lang_mode  # 'auto' | 'id' | 'en'
        self.symptoms=[]; self.rejected=[]; self.pending=None
        self.turn=0; self.lang='en'

    def reset(self):
        self.symptoms=[]; self.rejected=[]; self.pending=None
        self.turn=0  # NOTE: bahasa TIDAK direset agar konsisten dalam sesi

    # ── classification (RoBERTa) ──
    def _classify(self, mdl, syms):
        text = ', '.join(sorted(syms)) if syms else ''
        if not text: return None, [], 0.0
        enc = tokenizer(text, return_tensors='pt', max_length=MAX_LEN,
                        truncation=True, padding='max_length')
        with torch.no_grad():
            logits = mdl(enc['input_ids'].to(device),
                         enc['attention_mask'].to(device)).logits
        probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
        idx3  = probs.argsort()[-3:][::-1]
        top3  = [(id2label[int(i)], float(probs[i])) for i in idx3]
        return top3[0][0], top3, float(probs[idx3[0]])

    # ── retrieval (TF-IDF) ──
    def _retrieve(self, disease, query):
        sub = dialog_df[dialog_df['predicted_disease']==disease]
        if len(sub)==0: sub = dialog_df
        q = tfidf.transform([query])
        sims = cosine_similarity(q, tfidf_matrix[sub.index])[0]
        bi = sub.index[sims.argmax()]
        full = dialog_df.loc[bi,'doctor_text']
        return ' '.join(re.split(r'(?<=[.!?]) +', full.strip())[:3])

    def _next_question(self, top3):
        if not top3: return None
        d1=top3[0][0]; d2=top3[1][0] if len(top3)>1 else d1
        s1=disease_symptom_stats.get(d1,{}); s2=disease_symptom_stats.get(d2,{})
        best,score=None,-1
        for sym in SYMPTOMS_28:
            if sym in self.symptoms or sym in self.rejected: continue
            sc = s1.get(sym,0)-s2.get(sym,0)
            if sc>score: score,best=sc,sym
        return best

    # ── intent detection ──
    def _intent(self, text):
        t=text.lower().strip(); w=set(t.split())
        if any(rf in t for rf in RED_FLAG_WORDS):           return 'red_flag'
        if self.pending and (t in YES or w & YES):          return 'confirm_yes'
        if self.pending and (t in NO  or w & NO):           return 'confirm_no'
        if t in GREETING or w & GREETING:                   return 'greeting'
        if any(p in t for p in THANKS):                     return 'thanks'
        if t in BYE or any(p in t for p in BYE):            return 'bye'
        if extract_symptoms(text):                          return 'symptom'
        if w & HEALTH_CTX:                                  return 'health_q'
        return 'off_topic'

    def _diagnose(self, ID):
        pred, top3, conf = self._classify(model, self.symptoms)
        pb, _, cb        = self._classify(base_model_demo, self.symptoms)
        advice = self._retrieve(pred, ' '.join(self.symptoms))
        syms = [SYMPTOM_ID.get(s,s) for s in self.symptoms] if ID else list(self.symptoms)
        base_str = f"{pb} ({cb:.0%})" if pb else "—"
        t3 = '\n'.join([f"  {i+1}. {d} — {p:.0%}" for i,(d,p) in enumerate(top3)])
        self.reset()
        if ID:
            return (f"**Gejala terdeteksi:** {', '.join(syms)}\n\n"
                    f"**Kemungkinan penyakit:** {pred} (confidence: {conf:.0%})\n\n"
                    f"**3 kemungkinan teratas:**\n{t3}\n\n"
                    f"**Model dasar (tanpa fine-tuning):** {base_str}\n\n"
                    f"---\n**Saran medis (sumber dataset, EN):** {advice}\n\n"
                    f"*⚠️ Hanya untuk edukasi. Silakan konsultasi ke dokter.*")
        return (f"**Detected symptoms:** {', '.join(syms)}\n\n"
                f"**Predicted disease:** {pred} ({conf:.0%} confidence)\n\n"
                f"**Top 3 possibilities:**\n{t3}\n\n"
                f"**Base model (no fine-tuning):** {base_str}\n\n"
                f"---\n**Medical advice:** {advice}\n\n"
                f"*⚠️ Educational purposes only. Please consult a real doctor.*")

    # ── main ──
    def chat(self, user_input):
        self.turn += 1
        # language: pakai mode manual, atau deteksi hybrid per-pesan
        if self.lang_mode in ('id','en'):
            self.lang = self.lang_mode
        else:  # auto
            self.lang, _m = detect_lang_smart(user_input, prev=self.lang)
        ID = self.lang == 'id'

        intent = self._intent(user_input)

        if intent == 'red_flag':
            self.turn -= 1
            return ("🚨 Gejala yang kamu sebutkan bisa menandakan kondisi DARURAT. "
                    "Segera hubungi IGD terdekat atau layanan darurat 119. "
                    "Jangan tunda!") if ID else \
                   ("🚨 Your symptoms may indicate an EMERGENCY. "
                    "Please contact emergency services immediately. Don't wait!")

        if intent == 'greeting':
            self.turn -= 1
            return ("Halo! Saya asisten kesehatan berbasis AI. Ceritakan keluhan atau "
                    "gejala yang kamu rasakan, misalnya: \"saya demam dan batuk sejak 2 hari\".") if ID else \
                   ("Hello! I'm an AI health assistant. Describe your symptoms, e.g. "
                    "\"I have fever and cough for 2 days\".")

        if intent == 'off_topic':
            self.turn -= 1
            return ("Maaf, saya hanya bisa membantu seputar **kesehatan dan gejala penyakit**. "
                    "Coba ceritakan keluhan yang kamu rasakan ya.") if ID else \
                   ("Sorry, I can only help with **health and symptoms**. "
                    "Please describe what you're feeling.")

        if intent == 'thanks':
            self.turn -= 1
            return ("Sama-sama! Semoga lekas sembuh. Ada keluhan lain?") if ID else \
                   ("You're welcome! Get well soon. Any other symptoms?")

        if intent == 'bye':
            self.reset()
            return ("Baik, jaga kesehatan ya! 🙏") if ID else ("Take care! 🙏")

        pending_before = self.pending

        if intent == 'confirm_yes':
            if self.pending and self.pending not in self.symptoms:
                self.symptoms.append(self.pending)
            self.pending = None
        if intent == 'confirm_no':
                    if self.pending:
                        self.rejected.append(self.pending)
                        if self.pending in self.symptoms:
                            self.symptoms.remove(self.pending)
                    self.pending = None

        for s in extract_symptoms(user_input):
            if s in self.rejected:
                continue  # jangan tambah lagi gejala yang baru saja ditolak
            if s not in self.symptoms:
                self.symptoms.append(s)

        if intent == 'health_q' and not self.symptoms:
            self.turn -= 1
            advice = self._retrieve('Common Cold', user_input)  # general retrieval
            return (f"Ini pertanyaan kesehatan umum. Dari basis data konsultasi: {advice}\n\n"
                    "Kalau kamu punya gejala spesifik, ceritakan ya.") if ID else \
                   (f"General health question. From our consultation database: {advice}\n\n"
                    "If you have specific symptoms, please describe them.")

        if not self.symptoms:
            return ("Saya belum menangkap gejala spesifik. Coba sebutkan apa yang dirasakan "
                    "— misal demam, mual, nyeri dada?") if ID else \
                   ("I didn't catch specific symptoms. Try naming what you feel.")

        pred, top3, conf = self._classify(model, self.symptoms)
        should = (conf >= CONFIDENCE_THRESHOLD or len(self.symptoms) >= 5 or self.turn >= MAX_TURNS)

        if should:
            return self._diagnose(ID)
        nxt = self._next_question(top3)
        self.pending = nxt
        syms = [SYMPTOM_ID.get(s,s) for s in self.symptoms] if ID else list(self.symptoms)
        if nxt:
            ns = SYMPTOM_ID.get(nxt,nxt) if ID else nxt
            return (f"Sudah dicatat: **{', '.join(syms)}**.\n\n"
                    f"Untuk mempersempit diagnosis — apakah kamu juga mengalami **{ns}**?") if ID else \
                   (f"Noted: **{', '.join(syms)}**.\n\nDo you also experience **{ns}**?")
        return self._diagnose(ID)

bot = MedicalChatbot()
print("MedicalChatbot (multi-layer, bilingual) ready.")

# Test
for msgs in [["halo"], ["saya nyeri dada dan sesak napas"],
             ["I have vomiting and diarrhea"], ["siapa presiden indonesia"]]:
    bot.reset()
    print(f"\nUser: {msgs[0]}\nBot : {bot.chat(msgs[0])[:200]}")


MedicalChatbot (multi-layer, bilingual) ready.

User: halo
Bot : Halo! Saya asisten kesehatan berbasis AI. Ceritakan keluhan atau gejala yang kamu rasakan, misalnya: "saya demam dan batuk sejak 2 hari".

User: saya nyeri dada dan sesak napas
Bot : **Gejala terdeteksi:** sesak napas, nyeri dada

**Kemungkinan penyakit:** Heart Disease (confidence: 98%)

**3 kemungkinan teratas:**
  1. Heart Disease — 98%
  2. Bronchitis — 1%
  3. Asthma — 0%

**

User: I have vomiting and diarrhea
Bot : **Detected symptoms:** vomiting, diarrhea

**Predicted disease:** Food Poisoning (99% confidence)

**Top 3 possibilities:**
  1. Food Poisoning — 99%
  2. Diabetes — 0%
  3. Anemia — 0%

**Base model 

User: siapa presiden indonesia
Bot : Sorry, I can only help with **health and symptoms**. Please describe what you're feeling.


### 3.3 Launch Gradio demo
> Setelah cell ini dirun, buka **http://localhost:7860** di browser.  
> Ganti `share=False` ke `share=True` untuk public URL.

In [22]:
!pip uninstall gradio hf_gradio -y
!pip install "gradio==5.9.1"

Found existing installation: gradio 4.44.0
Uninstalling gradio-4.44.0:
  Successfully uninstalled gradio-4.44.0
Found existing installation: hf-gradio 0.4.1
Uninstalling hf-gradio-0.4.1:
  Successfully uninstalled hf-gradio-0.4.1
Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/57.2 MB ? eta -:--:--
    --------------------------------------- 1.3/57.2 MB 11.5 MB/s eta 0:00:05
   -- ------------------------------------- 3.4/57.2 MB 11.1 MB/s eta 0:00:05
   --- ------------------------------------ 5.0/57.2 MB 9.3 MB/s eta 0:00:06
   ---- ----------------------------------- 6.8/57.2 MB 9.3 MB/s eta 0:00:06
   ------ --------------------------------- 8.7/57.2 MB 9.2 MB/s eta 0:00:06
   ------- -------------------------------- 10.2/57.2 MB 9.0 MB/s eta 0:00:06
   -------- ------------------------------- 11.8/57.2 MB 8.8 MB/s eta 0:00:06
   --------- ------------------------------ 13.6/57.2 MB 8.9 MB/s eta 0:00:05
 


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [48]:
# Quick reload dari outputs/ (skip training)
import os, json, pickle
import scipy.sparse
import pandas as pd
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification

# Definisikan ulang paths kalau belum ada
BASE_DIR = os.getcwd()
SAVE_DIR = os.path.join(BASE_DIR, 'outputs')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open(os.path.join(SAVE_DIR, 'roberta_finetuned', 'label_mappings.json')) as f:
    mappings = json.load(f)
label2id    = mappings['label2id']
id2label    = {int(k): v for k, v in mappings['id2label'].items()}
labels_list = mappings['labels_list']
NUM_LABELS  = len(labels_list)

tokenizer    = RobertaTokenizer.from_pretrained(os.path.join(SAVE_DIR, 'roberta_finetuned'))
model        = RobertaForSequenceClassification.from_pretrained(
               os.path.join(SAVE_DIR, 'best_checkpoint')).to(device)
model.eval()

dialog_df    = pd.read_csv(os.path.join(SAVE_DIR, 'dialog_index.csv'))
tfidf        = pickle.load(open(os.path.join(SAVE_DIR, 'tfidf_vectorizer.pkl'), 'rb'))
tfidf_matrix = scipy.sparse.load_npz(os.path.join(SAVE_DIR, 'tfidf_matrix.npz'))

with open(os.path.join(SAVE_DIR, 'disease_symptom_stats.json')) as f:
    disease_symptom_stats = json.load(f)
with open(os.path.join(SAVE_DIR, 'metrics.json')) as f:
    metrics = json.load(f)

MAX_LEN = 64
SYMPTOMS_28 = [
    'chest pain','dizziness','sore throat','sneezing','weight loss','blurred vision',
    'nausea','vomiting','sweating','insomnia','diarrhea','depression','weight gain',
    'rash','swelling','headache','fever','fatigue','abdominal pain','muscle pain',
    'back pain','runny nose','tremors','anxiety','cough','joint pain',
    'shortness of breath','appetite loss'
]

print(f"Loaded. {NUM_LABELS} classes, {len(dialog_df):,} dialogs.")
print("Device:", device)

Loaded. 30 classes, 37,321 dialogs.
Device: cpu


In [56]:
!pip uninstall gradio gradio-client hf-gradio hf_gradio -y
!pip install "gradio==4.44.1"

Found existing installation: gradio 5.9.1
Uninstalling gradio-5.9.1:
  Successfully uninstalled gradio-5.9.1
Found existing installation: gradio_client 1.5.2
Uninstalling gradio_client-1.5.2:
  Successfully uninstalled gradio_client-1.5.2


Defaulting to user installation because normal site-packages is not writeable
  Using cached gradio_client-1.3.0-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/18.1 MB ? eta -:--:--
   - -------------------------------------- 0.5/18.1 MB 8.7 MB/s eta 0:00:03
   ------ --------------------------------- 3.1/18.1 MB 11.3 MB/s eta 0:00:02
   ------------ --------------------------- 5.5/18.1 MB 11.5 MB/s eta 0:00:02
   -------------- ------------------------- 6.6/18.1 MB 10.6 MB/s eta 0:00:02
   ---------------- ----------------------- 7.3/18.1 MB 8.4 MB/s eta 0:00:02
   ------------------ --------------------- 8.4/18.1 MB 7.8 MB/s eta 0:00:02
   ------------------------ --------------- 11.3/18.1 MB 8.6 MB/s eta 0:00:01
   ----------------------------- ---------- 13.4/18.1 MB 9.0 MB/s eta 0:00:01
   ----------------------------------- ---- 16.0/18.1 MB 9.4 MB/s eta 0:00:01
   ---------------------------------------  17.8/18.1 MB 9.4 MB/s eta 0:00:01
   --


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from IPython.display import display, clear_output
import ipywidgets as widgets

# Info perbandingan model (untuk presentasi)
ba  = metrics['base']['accuracy'];      bf  = metrics['base']['f1_macro']
fta = metrics['finetuned']['accuracy']; ftf = metrics['finetuned']['f1_macro']

header = widgets.HTML(
    f"<div style='padding:12px;background:#eef3f8;border-radius:8px;margin-bottom:8px;font-family:sans-serif'>"
    f"<b>🏥 Medical Symptom Chatbot</b> &nbsp;|&nbsp; RoBERTa fine-tuned<br>"
    f"<b>Base</b>: Acc {ba:.1%} / F1 {bf:.1%} &nbsp;→&nbsp; "
    f"<b>Fine-tuned</b>: Acc {fta:.1%} / F1 {ftf:.1%} &nbsp;|&nbsp; "
    f"30 penyakit &nbsp;|&nbsp; {len(dialog_df):,} dialog</div>"
)

lang_dd = widgets.Dropdown(
    options=[('Auto (deteksi otomatis)','auto'), ('Indonesia','id'), ('English','en')],
    value='auto', description='Bahasa:', layout=widgets.Layout(width='280px'))

bot = MedicalChatbot(lang_mode=lang_dd.value)
chat_log = []

output     = widgets.Output()
text_input = widgets.Text(placeholder='Ceritakan gejala kamu / describe your symptoms...',
                          layout=widgets.Layout(width='65%'))
send_btn   = widgets.Button(description='Send', button_style='primary')
reset_btn  = widgets.Button(description='Reset', button_style='warning')

def render():
    with output:
        clear_output()
        for u,b in chat_log:
            print(f"🧑 You : {u}")
            print(f"🤖 Bot : {b}")
            print("-"*70)

def on_send(b=None):
    msg = text_input.value.strip()
    if not msg: return
    text_input.value=''
    bot.lang_mode = lang_dd.value          # sinkron pilihan dropdown
    chat_log.append((msg, bot.chat(msg)))
    render()

def on_reset(b=None):
    bot.reset(); chat_log.clear()
    with output: clear_output()

def on_lang(change):
    bot.lang_mode = lang_dd.value

send_btn.on_click(on_send)
reset_btn.on_click(on_reset)
text_input.on_submit(on_send)
lang_dd.observe(on_lang, names='value')

display(header)
display(widgets.HBox([lang_dd]))
display(widgets.HBox([text_input, send_btn, reset_btn]))
display(output)

C:\Users\Milan\AppData\Local\Temp\ipykernel_20304\3079582340.py:54: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  text_input.on_submit(on_send)


HTML(value="<div style='padding:12px;background:#eef3f8;border-radius:8px;margin-bottom:8px;font-family:sans-s…

Output()